# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### ML Task Type: Scoring & Prioritized Ranking

To optimize our content-refresh workflow, we frame this problem as a **Scoring (Regression)** task that feeds directly into a **Ranking** system:

1. **The Scoring Task (Regression):**
   Our model will predict a continuous value the **Expected CTR** for a given page based on its position, query intent, and content metadata. By subtracting the page's actual, observed CTR from this predicted baseline, we calculate a custom metric: the **CTR Gap (Opportunity Score)**.

   <br>
   $$\text{Opportunity Score} = \text{Predicted Expected CTR} - \text{Actual Observed CTR}$$
   <br>

2. **The Ranking Task:**
   Once every page has an Opportunity Score, we sort the pages in descending order. This produces a prioritized queue.

* **Why this beats alternative ML framings:**
  * **Instead of Classification:** Binary classification ("Needs Update" vs. "Keep") lacks granularity. Scoring allows us to quantify *how much* a page is underperforming, preventing editorial teams from wasting time on pages with negligible room for improvement.
  * **Instead of Clustering:** Unsupervised clustering might group similar pages together, but it cannot rank them by potential traffic recovery or measure execution priority.

In [1]:
# =====================================================================
# SYSTEM CONFIGURATION: TASK TYPE DEFINITION
# =====================================================================

pipeline_config = {
    "primary_ml_task": "Scoring (Regression)",
    "downstream_application": "Prioritized Ranking Queue",
    "target_variable": "ctr_gap",
    "mathematical_definition": "expected_ctr - observed_ctr",
    "output_action": "Generate a sorted backlog of pages for copywriting optimization"
}

# Print configuration diagnostics to verify execution
print("✅ ML Task Type successfully registered in notebook session.")
print(f"🔹 Task Type : {pipeline_config['primary_ml_task']}")
print(f"🔹 Downstream: {pipeline_config['downstream_application']}")
print(f"🔹 Target Col: {pipeline_config['target_variable']}")

✅ ML Task Type successfully registered in notebook session.
🔹 Task Type : Scoring (Regression)
🔹 Downstream: Prioritized Ranking Queue
🔹 Target Col: ctr_gap


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target and Proxy Formulation

To model this, we define a clear relationship between our predicted target and our business proxy:

1. **What we predict (The Machine Learning Target):**
   Our model will predict **Expected CTR** ($Expected\_CTR$). This is a continuous numerical target.
   
2. **Where the label comes from:**
   * **Expected CTR (Observed Outcome):** The training labels come directly from the historical, observed `ctr` column in our dataset. Our model learns what a "normal" CTR looks like by training on real search behavior across thousands of pages.
   * **CTR Gap / Opportunity Score (Defined Rule):** Once our model predicts the $Expected\_CTR$, we apply a post-prediction mathematical rule to calculate the **CTR Gap**:
   <br>
     $$\text{CTR Gap} = \text{Predicted Expected CTR} - \text{Actual Observed CTR}$$
   <br>
* **Why we use this proxy framework:**
  Using the raw `ctr` alone as a target doesn't tell us if a page is underperforming. A $0.5\%$ CTR is outstanding for position 15, but horrific for position 2. By predicting a position-and-metadata-adjusted $Expected\_CTR$ first, our proxy ($CTR\_Gap$) normalizes the metric, isolating true content performance from simple ranking placement.

In [2]:
# =====================================================================
# SYSTEM CONFIGURATION: TARGET AND PROXY SCHEMAS
# =====================================================================

# 1. Define the targets and proxies as a structured schema.
# In production, keeping schemas explicit prevents "target leakage"
# (using features that won't be available at prediction time).
target_schema = {
    "model_prediction_target": {
        "column_name": "ctr",
        "type": "observed_outcome",
        "data_type": "float64",
        "description": "The actual Click-Through Rate recorded in Google Search Console."
    },
    "business_proxy_target": {
        "column_name": "ctr_gap",
        "type": "derived_rule",
        "data_type": "float64",
        "description": "The difference between predicted expected CTR and actual observed CTR."
    }
}

# 2. Programmatically verify our schema definitions against the target logic
# We want to ensure that our business proxy rules make mathematical sense.
def verify_proxy_logic(expected_ctr, actual_ctr):
    """
    Simulates the proxy logic.
    If actual CTR is lower than expected, the gap must be positive (opportunity).
    If actual CTR is higher than expected, the gap is negative (outperformer).
    """
    gap = expected_ctr - actual_ctr
    return gap

# Let's test a hypothetical scenario to verify the rule:
# Scenario: A page in position 2 typically gets a 5.0% expected CTR, but only gets 1.2% actual CTR.
test_gap = verify_proxy_logic(expected_ctr=5.0, actual_ctr=1.2)

print("--- 🎯 Target & Proxy Verification ---")
print(f"🔹 ML Target Variable   : {target_schema['model_prediction_target']['column_name']} ({target_schema['model_prediction_target']['type']})")
print(f"🔹 Business Proxy Target: {target_schema['business_proxy_target']['column_name']} ({target_schema['business_proxy_target']['type']})")
print(f"🔹 Test Scenario (Expected 5.0% vs. Actual 1.2%) -> Calculated Opportunity Gap: {test_gap:+.2f}%")

--- 🎯 Target & Proxy Verification ---
🔹 ML Target Variable   : ctr (observed_outcome)
🔹 Business Proxy Target: ctr_gap (derived_rule)
🔹 Test Scenario (Expected 5.0% vs. Actual 1.2%) -> Calculated Opportunity Gap: +3.80%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.